# Hand-Gesture-Control Training Pipeline

This notebook trains a gesture recognition model using the HaGRID dataset.

## Pipeline Overview
1. **Setup** - Clone repo, install dependencies, mount Drive
2. **Data** - Download HaGRID 30k from Kaggle, prepare splits
3. **Train** - Train EfficientNet-B0 classifier
4. **Evaluate** - Test accuracy and per-class metrics
5. **Export** - Save model to Google Drive for local inference

## Storage Strategy
```
/content/data/          ← HaGRID dataset (temp, ~3GB for 30k)
/content/drive/MyDrive/ ← Trained model only (~20MB, persistent)
```

---

## 0) Enable GPU (Required)

**Before running any cells:**
1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator** to **GPU** (T4 is fine)
3. Click **Save**

Run the cell below to verify GPU is available:

In [ ]:
# Verify GPU is available
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU available: {gpu_name}")
    print(f"   CUDA version: {torch.version.cuda}")
else:
    print("No GPU detected!")
    print("   Go to Runtime → Change runtime type → GPU")

---
## 1) Setup: Clone Repo & Install Dependencies

In [ ]:
# Clone the repository
REPO_URL = "https://github.com/sterlingwalker/Hand-Gesture-Control.git"
REPO_DIR = "Hand-Gesture-Control"

!git clone {REPO_URL}
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# Install dependencies
!pip install -q -U pip
!pip install -q -r requirements.txt

# Verify key imports
import torch
import torchvision
import cv2
import mediapipe as mp

print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"MediaPipe: {mp.__version__}")

In [ ]:
# Mount Google Drive (for saving trained models)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Create output directory for models
MODELS_DIR = Path('/content/drive/MyDrive/hand-gesture-models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Models will be saved to: {MODELS_DIR}")

---
## 2) Data: Download HaGRID 30k from Kaggle

### Kaggle API Setup
You need a Kaggle account and API token:
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to **API** section
3. Click **Create New Token**
4. Copy your **username** and **API key** when prompted below

In [ ]:
# Install Kaggle CLI and set up credentials
!pip install -q kaggle

import os
import json
from pathlib import Path

# Check if kaggle.json already exists (from previous session)
kaggle_dir = Path.home() / '.kaggle'
kaggle_json = kaggle_dir / 'kaggle.json'

if kaggle_json.exists():
    print("Kaggle credentials already configured")
else:
    print("Enter your Kaggle credentials (from kaggle.com/settings → API → Create New Token):\n")
    username = input("Kaggle username: ").strip()
    api_key = input("Kaggle API key: ").strip()
    
    # Create kaggle.json
    kaggle_dir.mkdir(exist_ok=True)
    credentials = {"username": username, "key": api_key}
    kaggle_json.write_text(json.dumps(credentials))
    os.chmod(kaggle_json, 0o600)
    print("\nKaggle credentials configured")

In [ ]:
# Download HaGRID 30k sample (~3GB)
# This goes to Colab temp storage (resets each session, but that's OK)

from pathlib import Path

RAW_DIR = Path('/content/data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download from Kaggle
!kaggle datasets download -d innominate817/hagrid-sample-30k-384p -p {RAW_DIR}

# Unzip
!unzip -q {RAW_DIR}/hagrid-sample-30k-384p.zip -d {RAW_DIR}/hagrid

# Show what we got
!ls -la {RAW_DIR}/hagrid/
print(f"\nHaGRID 30k downloaded to {RAW_DIR}/hagrid/")

In [ ]:
# Explore the dataset structure - diagnostic
from pathlib import Path
import os

hagrid_base = Path('/content/data/raw/hagrid')

print("All directories found:")
print("=" * 60)

all_dirs = []
for root, dirs, files in os.walk(hagrid_base):
    level = root.replace(str(hagrid_base), '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    n_images = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    all_dirs.append((root, folder_name, n_images, level))
    if level < 4:  # Only show first 4 levels
        if n_images > 0:
            print(f"{indent}{folder_name}/ ({n_images} images)")
        else:
            print(f"{indent}{folder_name}/")

print("\n" + "=" * 60)
print("\nAll unique folder names with images:")
folders_with_images = set()
for root, name, n_images, level in all_dirs:
    if n_images > 0:
        folders_with_images.add(name)
print(sorted(folders_with_images))

print("\n" + "=" * 60)
print("\nSample of actual files:")
all_images = list(hagrid_base.rglob("*.jpg"))[:5]
for img in all_images:
    print(f"  {img.relative_to(hagrid_base)}")

In [ ]:
# Prepare dataset: extract only our 7 target gestures, create train/val/test splits

# Our target gestures (Our name -> HaGRID class name)
# OPEN_PALM  -> palm
# FIST       -> fist  
# THUMBS_UP  -> like
# THUMBS_DOWN -> dislike
# OK_SIGN    -> ok
# PEACE      -> peace
# POINTING   -> one

TARGET_GESTURES = "palm,fist,like,dislike,ok,peace,one"

from pathlib import Path

# Auto-detect the correct raw directory by finding where gesture folders actually are
hagrid_base = Path('/content/data/raw/hagrid')
raw_dir = None

# Strategy 1: Look for a directory containing our target gesture folders
target_list = TARGET_GESTURES.split(',')
for candidate in [hagrid_base] + list(hagrid_base.glob('*')) + list(hagrid_base.glob('*/*')):
    if not candidate.is_dir():
        continue
    subdirs = [d.name for d in candidate.iterdir() if d.is_dir()]
    matches = [t for t in target_list if t in subdirs]
    if len(matches) >= 3:  # Found at least 3 of our target gestures
        raw_dir = candidate
        print(f"Found gesture folders in: {raw_dir}")
        print(f"Matched gestures: {matches}")
        break

# Strategy 2: Check if structure is train/val/test with gesture subfolders
if raw_dir is None:
    for candidate in [hagrid_base] + list(hagrid_base.glob('*')):
        if not candidate.is_dir():
            continue
        if (candidate / 'train').exists():
            train_subdirs = [d.name for d in (candidate / 'train').iterdir() if d.is_dir()]
            matches = [t for t in target_list if t in train_subdirs]
            if len(matches) >= 3:
                raw_dir = candidate
                print(f"Found train/val/test structure in: {raw_dir}")
                print(f"Matched gestures in train/: {matches}")
                break

if raw_dir is None:
    print("ERROR: Could not auto-detect raw directory!")
    print("\nAvailable directories:")
    for p in sorted(hagrid_base.rglob('*'))[:30]:
        if p.is_dir():
            print(f"  {p.relative_to(hagrid_base)}")
    raise SystemExit("Please check dataset structure and set raw_dir manually")

print(f"\nUsing raw directory: {raw_dir}")

# Run the preparation script (use copy instead of symlink for Colab compatibility)
!python scripts/prepare_hagrid_subset.py \
    --raw-dir {raw_dir} \
    --out-dir /content/data/processed/hagrid \
    --classes {TARGET_GESTURES} \
    --val 0.1 \
    --test 0.1 \
    --seed 42 \
    --link-type copy \
    --overwrite

In [ ]:
# Verify the prepared dataset
from pathlib import Path

processed_dir = Path('/content/data/processed/hagrid')

print("Prepared dataset summary:\n")
for split in ['train', 'val', 'test']:
    split_dir = processed_dir / split
    if split_dir.exists():
        classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
        total = sum(len(list((split_dir / c).glob('*'))) for c in classes)
        print(f"{split}: {total} images across {len(classes)} classes")
        if split == 'train':
            print(f"   Classes: {', '.join(classes)}")

---
## 3) Train: EfficientNet-B0 Gesture Classifier

Training configuration:
- **Model**: EfficientNet-B0 (pretrained, frozen backbone)
- **Optimizer**: AdamW
- **Epochs**: 15 (adjust as needed)
- **Batch size**: 64 (good for T4 GPU)

In [ ]:
# Train the model
# Best checkpoint will be saved to models/hagrid_efficientnet.pt

# Ensure models directory exists
!mkdir -p /content/Hand-Gesture-Control/models

!PYTHONPATH=/content/Hand-Gesture-Control python scripts/train_hagrid.py \
    --data-dir /content/data/processed/hagrid \
    --output /content/Hand-Gesture-Control/models/hagrid_efficientnet.pt \
    --epochs 15 \
    --batch-size 64 \
    --lr 0.001

---
## 4) Evaluate: Test Set Performance

In [ ]:
# Evaluate on test set
!PYTHONPATH=/content/Hand-Gesture-Control python scripts/eval_hagrid.py \
    --data-dir /content/data/processed/hagrid \
    --checkpoint /content/Hand-Gesture-Control/models/hagrid_efficientnet.pt

In [ ]:
# Detailed evaluation with confusion matrix
!pip install -q scikit-learn seaborn

import torch
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, '/content/Hand-Gesture-Control')

from src.hand_gesture_control.model import load_checkpoint
from src.hand_gesture_control.data import build_dataloaders
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Load model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, meta = load_checkpoint('/content/Hand-Gesture-Control/models/hagrid_efficientnet.pt')
model = model.to(device)
model.eval()

# Load test data
_, _, test_loader = build_dataloaders(
    '/content/data/processed/hagrid',
    batch_size=64,
    image_size=meta.image_size
)

# Collect predictions
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Classification report
print("Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=meta.class_names))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=meta.class_names,
            yticklabels=meta.class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

---
## 5) Export: Save Model to Google Drive

Copy the trained model to Google Drive so it persists after the Colab session ends.

Then download it to your local machine for real-time webcam inference.

In [ ]:
# Copy model to Google Drive
import shutil
from pathlib import Path
from datetime import datetime

# Source and destination
src_model = Path('/content/Hand-Gesture-Control/models/hagrid_efficientnet.pt')
drive_dir = Path('/content/drive/MyDrive/hand-gesture-models')
drive_dir.mkdir(parents=True, exist_ok=True)

# Copy with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
dst_model = drive_dir / f'hagrid_efficientnet_{timestamp}.pt'
dst_latest = drive_dir / 'hagrid_efficientnet_latest.pt'

shutil.copy(src_model, dst_model)
shutil.copy(src_model, dst_latest)

print(f"Model saved to Google Drive:")
print(f"   {dst_model}")
print(f"   {dst_latest}")
print(f"\nDownload 'hagrid_efficientnet_latest.pt' to your local machine")
print(f"   Place it in: Hand-Gesture-Control/models/")

In [ ]:
# Alternative: Download directly from Colab
from google.colab import files

print("Downloading model to your computer...")
files.download('/content/Hand-Gesture-Control/models/hagrid_efficientnet.pt')

---
## 6) Local Inference (Run on Your Machine)

After downloading the model, run this on your local machine:

```bash
# Navigate to repo
cd Hand-Gesture-Control

# Run webcam demo
python scripts/predict_webcam.py --checkpoint models/hagrid_efficientnet.pt
```

Press `q` to quit the demo.

---
## 7) Next Steps

### If accuracy < 85%: Scale up dataset
```python
# Download HaGRID 120k instead (run this instead of 30k download)
!kaggle datasets download -d innominate817/hagrid-sample-120k-384p -p /content/data/raw
```

### Gesture Mapping Reference
| Our Gesture | HaGRID Class | Action |
|-------------|--------------|--------|
| OPEN_PALM | palm | Idle |
| FIST | fist | Click |
| THUMBS_UP | like | Confirm (Enter) |
| THUMBS_DOWN | dislike | Cancel (Escape) |
| OK_SIGN | ok | Mode Switch |
| PEACE | peace | (Reserved) |
| POINTING | one | Cursor Control |

### Checklist
- [ ] Model achieves >85% accuracy
- [ ] Model downloaded to local machine
- [ ] Webcam demo runs locally
- [ ] Ready for gesture state machine implementation